# 👹 FaceDig TPS — Analisis Geometrik Morfometrik Wajah
**Pipeline:** Procrustes → PCA → Uji Statistik (Permutation MANOVA)

| Folder | Isi |
|--------|-----|
| `2023/`, `2024/`, `2025/` | Foto formal cowok & cewek + file `.tps` dari FaceDig |

---


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Ellipse
from scipy import stats
from sklearn.decomposition import PCA
from PIL import Image
import os, re, warnings
warnings.filterwarnings('ignore')

# Cek versi library
import importlib
for lib in ['numpy','pandas','matplotlib','sklearn','PIL','scipy']:
    try:
        mod = importlib.import_module(lib)
        ver = getattr(mod, '__version__', '?')
        print(f"  ✅ {lib:<15} {ver}")
    except ImportError:
        print(f"  ❌ {lib} — install dulu: pip install {lib}")
print("\n✅ Setup selesai!")

## ⚙️ Konfigurasi — Sesuaikan di sini sebelum run

In [ ]:
BASE_DIR   = "Data"       # Folder utama (berisi 2023/, 2024/, 2025/)
YEARS      = ["2023", "2024", "2025"]

# Cara identifikasi jenis kelamin dari nama file
# L_ = Laki-laki, P_ = Perempuan
SEX_FROM   = "filename"

# Kata kunci untuk identifikasi jenis kelamin
MALE_KW    = ["l_", "male", "laki", "pria", "cowok"]
FEMALE_KW  = ["p_", "female", "perempuan", "wanita", "cewek"]

# Output directories
OUT_FIG    = "output/figures"
OUT_TBL    = "output/tables"
IMG_COORD_ORIGIN = "bottom-left"  # Set ke "top-left" jika landmark TPS sudah menggunakan origin kiri-atas
os.makedirs(OUT_FIG, exist_ok=True)
os.makedirs(OUT_TBL, exist_ok=True)

# Warna & marker per grup
COLOR      = {"male": "#2979FF", "female": "#F50057"}
MARKER     = {"2023": "o", "2024": "s", "2025": "^"}
YEAR_CLR   = {"2023": "#FF6D00", "2024": "#6200EA", "2025": "#00BFA5"}

print(f"📁 Base dir : {os.path.abspath(BASE_DIR)}")
print(f"📅 Tahun    : {YEARS}")
print(f"🔍 Sex dari : {SEX_FROM}")

## 📄 TPS Parser
TPS Parser adalah perangkat lunak atau library yang digunakan untuk membaca dan menguraikan (parsing) data dari file TPS (TopSpeed

In [ ]:
def parse_tps(filepath):
    """
    Baca file .tps dari FaceDig.
    Format standar:
        LM=N
        x1 y1
        x2 y2
        ...
        IMAGE=nama_file.jpg
        ID=1
        SCALE=1.0   (opsional)
    Mengembalikan list of dict: {landmarks, image, id, scale}
    """
    specimens, cur = [], {"landmarks": [], "image": None, "id": None, "scale": 1.0}
    n_lm, reading, count = 0, False, 0

    with open(filepath, encoding="utf-8", errors="ignore") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            UP = line.upper()

            if UP.startswith("LM="):
                if cur["landmarks"]:
                    specimens.append({**cur, "landmarks": np.array(cur["landmarks"])})
                cur = {"landmarks": [], "image": None, "id": None, "scale": 1.0}
                n_lm, reading, count = int(line.split("=")[1]), True, 0

            elif reading and count < n_lm:
                pts = line.split()
                if len(pts) >= 2:
                    try:
                        cur["landmarks"].append([float(pts[0]), float(pts[1])])
                        count += 1
                    except ValueError:
                        pass
                if count >= n_lm:
                    reading = False

            elif UP.startswith("IMAGE="):
                cur["image"] = line.split("=", 1)[1].strip()
            elif UP.startswith("ID="):
                cur["id"] = line.split("=", 1)[1].strip()
            elif UP.startswith("SCALE="):
                try: cur["scale"] = float(line.split("=")[1])
                except ValueError: pass

    if cur["landmarks"]:
        specimens.append({**cur, "landmarks": np.array(cur["landmarks"])})
    return specimens


def detect_sex(filename, subfolder=""):
    """Deteksi jenis kelamin dari nama file atau subfolder."""
    text = (filename + " " + subfolder).lower()
    for kw in FEMALE_KW:          # cek female dulu (menghindari 'female' ke-detect 'male')
        if kw in text: return "female"
    for kw in MALE_KW:
        if kw in text: return "male"
    return "unknown"


print("✅ Parser TPS siap!")

## 📂 Load Data dari Semua Folder

In [ ]:
def load_all_data(base_dir, years):
    rows = []
    for year in years:
        year_dir = os.path.join(base_dir, year)
        if not os.path.exists(year_dir):
            print(f"\u26a0\ufe0f  Folder '{year}' tidak ditemukan — skip"); continue

        tps_files = []
        for root, _, files in os.walk(year_dir):
            for f in files:
                if f.lower().endswith(".tps"):
                    tps_files.append(os.path.join(root, f))

        if not tps_files:
            print(f"\u26a0\ufe0f  Tidak ada .tps di folder {year}"); continue

        print(f"\n\U0001f4c1 {year} \u2192 {len(tps_files)} file TPS")

        for tps_path in tps_files:
            specs = parse_tps(tps_path)
            tps_dir = os.path.dirname(tps_path)
            subfolder = os.path.relpath(tps_dir, year_dir)

            for sp in specs:
                img_name = sp["image"] or ""
                img_path = None
                for candidate in [
                    os.path.join(tps_dir, img_name),
                    os.path.join(tps_dir, os.path.basename(img_name)),
                    os.path.join(year_dir, img_name),
                ]:
                    if os.path.exists(candidate):
                        img_path = candidate; break

                sex = detect_sex(
                    img_name,
                    subfolder if SEX_FROM == "subfolder" else ""
                )

                rows.append({
                    "year": year, "sex": sex,
                    "image_name": img_name, "image_path": img_path,
                    "landmarks": sp["landmarks"],
                    "n_lm": len(sp["landmarks"]),
                    "scale": sp["scale"],
                    "tps_file": tps_path,
                })

    df = pd.DataFrame(rows)
    if df.empty:
        print("\u274c Tidak ada data ditemukan. Periksa path dan format TPS!")
        return df

    print(f"\n{'='*50}")
    print(f"\U0001f4ca TOTAL SPECIMEN: {len(df)}")
    print(df.groupby(["year","sex"]).size().unstack(fill_value=0).to_string())

    unknown = df[df["sex"]=="unknown"]
    if len(unknown):
        print(f"\n\u26a0\ufe0f  {len(unknown)} specimen tidak terdeteksi jenis kelaminnya:")
        for _, r in unknown.iterrows():
            print(f"   - {r['image_name']} ({r['year']})")
    return df

df_raw = load_all_data(BASE_DIR, YEARS)
df_raw.head()



## 🔄 Generalized Procrustes Analysis (GPA)
Generalized Procrustes Analysis (GPA) adalah metode analisis statistik multivariat yang digunakan untuk membandingkan, mencocokkan, dan menganalisis bentuk (shape) dari beberapa objek atau kumpulan data yang berbeda

In [ ]:
# ── Fungsi GPA ──────────────────────────────────────────────────────────────
def centroid_size(lm):
    c = lm - lm.mean(0)
    return np.sqrt((c**2).sum())
def normalize(lm):
    c = lm - lm.mean(0)
    cs = centroid_size(lm)
    return c / cs, cs

def optimal_rotation(src, tgt):
    """Rotasi src agar paling mirip tgt (SVD, tanpa refleksi)."""
    M = tgt.T @ src
    U, S, Vt = np.linalg.svd(M)
    D = np.eye(len(S)); D[-1,-1] = np.sign(np.linalg.det(U @ Vt))
    return src @ (U @ D @ Vt).T

def gpa(lm_list, max_iter=200, tol=1e-10):
    """
    Generalized Procrustes Analysis.
    Input : list of (n_lm, 2) arrays
    Output: (aligned list, mean shape, centroid sizes)
    """
    aligned = []
    cs_list = []
    for lm in lm_list:
        a, cs = normalize(lm)
        aligned.append(a)
        cs_list.append(cs)

    mean_shape = aligned[0].copy()
    for it in range(max_iter):
        old = mean_shape.copy()
        aligned = [optimal_rotation(a, mean_shape) for a in aligned]
        mean_shape, _ = normalize(np.mean(aligned, 0))
        if np.sqrt(((mean_shape - old)**2).sum()) < tol:
            print(f"   GPA konvergen → {it+1} iterasi"); break
    return aligned, mean_shape, np.array(cs_list)

# ── Jalankan GPA ─────────────────────────────────────────────────────────────
print("🔄 Menjalankan GPA...")

# Filter valid
df = df_raw[df_raw["n_lm"] > 0].copy().reset_index(drop=True)

# Pastikan jumlah landmark konsisten
n_mode = df["n_lm"].mode()[0]
if df["n_lm"].nunique() > 1:
    print(f"⚠️  Jumlah landmark tidak seragam. Menggunakan n={n_mode}")
    df = df[df["n_lm"] == n_mode].reset_index(drop=True)

lm_list = list(df["landmarks"])
aligned, mean_shape, cs_arr = gpa(lm_list)

df["aligned"] = aligned
df["cs"] = cs_arr

print(f"✅ GPA selesai! {len(df)} specimen, {n_mode} landmark")
print(f"\n📊 Distribusi final:")
print(df.groupby(["year","sex"]).size().unstack(fill_value=0))


## 📸 Contoh Visualisasi Landmark di Atas Foto

In [ ]:
def draw_landmarks(ax, img_path, lm, title="", clr="red", nums=True):
    """Plot titik landmark di atas foto (atau koordinat saja jika foto tidak ada)."""
    if img_path and os.path.exists(img_path):
        img = np.array(Image.open(img_path))
        h = img.shape[0]
        ax.imshow(img)
        ys = lm[:, 1] if IMG_COORD_ORIGIN == "top-left" else h - lm[:, 1]
        xs = lm[:, 0]
    else:
        xs, ys = lm[:, 0], -lm[:, 1]
        ax.set_facecolor("#1a1a2e")

    ax.scatter(xs, ys, c=clr, s=60, zorder=5,
               edgecolors="white", linewidths=0.6, alpha=0.9)
    if nums:
        for i, (x, y) in enumerate(zip(xs, ys)):
            ax.annotate(str(i+1), (x, y), xytext=(3,3),
                        textcoords="offset points",
                        fontsize=6, color="white", fontweight="bold")
    ax.set_title(title, fontsize=9, fontweight="bold", pad=4)
    ax.axis("off")


# Plot contoh 1 foto per kelompok (tahun × sex)
rows_to_show = [("male",   COLOR["male"]),
                ("female", COLOR["female"])]
row_labels   = ["Laki-laki", "Perempuan"]

fig, axes = plt.subplots(2, len(YEARS), figsize=(5*len(YEARS), 9))
fig.patch.set_facecolor("#0d1117")
fig.suptitle("Visualisasi Landmark Wajah (Contoh Specimen)",
             fontsize=14, color="white", fontweight="bold", y=1.01)

for ri, (sex, clr) in enumerate(rows_to_show):
    for ci, yr in enumerate(YEARS):
        ax = axes[ri, ci] if len(YEARS) > 1 else axes[ri]
        sub = df[(df.sex==sex) & (df.year==yr)]
        if sub.empty:
            ax.text(0.5, 0.5, "Tidak\nada data",
                    ha="center", va="center", color="gray",
                    transform=ax.transAxes); ax.axis("off"); continue
        s = sub.iloc[0]
        draw_landmarks(ax, s.image_path, s.landmarks,
                       title=f"{row_labels[ri]} — {yr}  (n={len(sub)})",
                       clr=clr)

plt.tight_layout()
plt.savefig(f"{OUT_FIG}/01_landmark_foto.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print("✅ Saved: 01_landmark_foto.png")



## 📐 Mean Shape: Laki-laki vs Perempuan per Tahun

In [ ]:
# Menghitung dan visualisasi Mean Shape per Tahun untuk Laki-laki dan Perempuan
fig, axes = plt.subplots(1, len(YEARS), figsize=(6*len(YEARS), 6))
if len(YEARS) == 1: axes = [axes]

for i, yr in enumerate(YEARS):
    ax = axes[i]
    m_sub = df[(df.sex=="male") & (df.year==yr)]
    f_sub = df[(df.sex=="female") & (df.year==yr)]
    
    if not m_sub.empty and not f_sub.empty:
        m_mean = np.mean(list(m_sub.aligned), axis=0)
        f_mean = np.mean(list(f_sub.aligned), axis=0)
        
        # Tanda minus (-) pada sumbu Y SUDAH DIHILANGKAN agar wajah tidak terbalik
        ax.scatter(m_mean[:,0], m_mean[:,1], c=COLOR["male"], s=80, label="Laki-laki", edgecolors="white", zorder=3)
        ax.scatter(f_mean[:,0], f_mean[:,1], c=COLOR["female"], s=80, label="Perempuan", edgecolors="white", zorder=3)
        
        # Garis penghubung (vektor perubahan antar kelamin)
        for j in range(len(m_mean)):
            ax.plot([m_mean[j,0], f_mean[j,0]], [m_mean[j,1], f_mean[j,1]], color="gray", alpha=0.3)
            
        ax.set_title(f"Tahun {yr} (n Laki={len(m_sub)}, n Pr={len(f_sub)})", fontweight="bold")
    else:
        ax.text(0.5, 0.5, f"Data tidak lengkap\nuntuk tahun {yr}", ha="center", va="center")
        
    ax.axis("equal")
    ax.axis("off")
    if i == 0: ax.legend(loc="upper right")

plt.suptitle("Rata-rata Bentuk Wajah (Mean Shape): Laki-laki vs Perempuan per Tahun", fontsize=15, fontweight="bold", y=0.95)
plt.tight_layout()
plt.savefig(f"{OUT_FIG}/02_mean_shape_year.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Tersimpan: {OUT_FIG}/02_mean_shape_year.png")


## 📊 Principal Component Analysis (PCA)

#### Matrix shape

In [ ]:
# Flatten aligned landmarks → matrix (n_specimen × n_variables)
X = np.vstack([a.flatten() for a in df.aligned])
print(f"Matrix shape: {X.shape}  ({X.shape[0]} specimen × {X.shape[1]} variabel)")

pca = PCA()
scores = pca.fit_transform(X)
expvar = pca.explained_variance_ratio_ * 100

df["PC1"] = scores[:,0]
df["PC2"] = scores[:,1]
df["PC3"] = scores[:,2]

print(f"\n📊 Varians yang dijelaskan:")
cumul = 0
for i in range(min(10, len(expvar))):
    cumul += expvar[i]
    print(f"   PC{i+1:2d}: {expvar[i]:6.2f}%  (kumulatif: {cumul:6.2f}%)")


In [ ]:
# ── PCA biplot sederhana ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
for sex, clr in [("male", COLOR["male"]), ("female", COLOR["female"])]:
    sub = df[df.sex==sex]
    lbl = "Laki-laki" if sex=="male" else "Perempuan"
    ax.scatter(sub.PC1, sub.PC2, c=clr, s=70, alpha=0.8,
               edgecolors="white", lw=0.5, label=lbl, zorder=3)
    # 95% confidence ellipse
    if len(sub) > 2:
        cov = np.cov(sub.PC1, sub.PC2)
        vals, vecs = np.linalg.eigh(cov)
        order = vals.argsort()[::-1]
        vals, vecs = vals[order], vecs[:,order]
        angle = np.degrees(np.arctan2(*vecs[:,0][::-1]))
        w, h = 2 * 1.96 * np.sqrt(vals)
        ell = Ellipse((sub.PC1.mean(), sub.PC2.mean()), w, h, angle=angle,
                      fc=clr, alpha=0.12, ec=clr, lw=2)
        ax.add_patch(ell)

ax.axhline(0, ls="--", color="gray", alpha=0.3, lw=0.8)
ax.axvline(0, ls="--", color="gray", alpha=0.3, lw=0.8)
ax.set_xlabel(f"PC1 ({expvar[0]:.1f}%)"); ax.set_ylabel(f"PC2 ({expvar[1]:.1f}%)")
ax.set_title("PCA — Laki-laki vs Perempuan (95% elips)", fontweight="bold")
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUT_FIG}/03_pca_sex.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: 03_pca_sex.png")



### 1. PCA Laki-Laki Suku Asal Seluruh Angkatan untuk melihat pengelompokan Suku

In [ ]:
# Ekstrak Suku Asal dari nama file (format: L_Tahun_No_SUKU.jpg)
def extract_suku(filename):
    if not isinstance(filename, str) or not filename: return "Unknown"
    name = os.path.splitext(filename)[0]
    parts = name.split('_')
    if len(parts) >= 4:
        return parts[-1].strip().upper()
    return "Unknown"

df['suku'] = df['image_name'].apply(extract_suku)

# 1. Plot PCA Laki-Laki Suku Asal
df_male = df[df['sex'] == 'male'].copy()

fig, ax = plt.subplots(figsize=(10, 7))
suku_list = df_male['suku'].unique()
cmap = plt.cm.get_cmap('tab20', len(suku_list))

for i, suku in enumerate(suku_list):
    sub = df_male[df_male['suku'] == suku]
    ax.scatter(sub.PC1, sub.PC2, s=120, alpha=0.8, edgecolors="white", lw=1, 
               color=cmap(i), label=f"{suku} (n={len(sub)})", zorder=3)
    
    # Tambahkan nama suku di dekat titik agar lebih mudah dibaca
    for _, row in sub.iterrows():
        ax.text(row.PC1 + 0.001, row.PC2 + 0.001, suku, fontsize=8, alpha=0.7)

ax.axhline(0, ls="--", color="gray", alpha=0.3, lw=0.8)
ax.axvline(0, ls="--", color="gray", alpha=0.3, lw=0.8)
ax.set_xlabel(f"PC1 ({expvar[0]:.1f}%)")
ax.set_ylabel(f"PC2 ({expvar[1]:.1f}%)")
ax.set_title("PCA Laki-Laki Berdasarkan Suku Asal (Seluruh Angkatan)", fontweight="bold", fontsize=13)

ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="Suku Asal")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_FIG}/04b_pca_male_suku.png", dpi=150, bbox_inches="tight")
plt.show()


### 2. PCA Perempuan Suku Asal Seluruh Angkatan untuk melihat pengelompokan Suku

In [ ]:
# 2. Plot PCA Perempuan Suku Asal
df_female = df[df['sex'] == 'female'].copy()

fig, ax = plt.subplots(figsize=(10, 7))
suku_list = df_female['suku'].unique()
cmap = plt.cm.get_cmap('tab20', len(suku_list))

for i, suku in enumerate(suku_list):
    sub = df_female[df_female['suku'] == suku]
    ax.scatter(sub.PC1, sub.PC2, s=120, alpha=0.8, edgecolors="white", lw=1, 
               color=cmap(i), marker='^', label=f"{suku} (n={len(sub)})", zorder=3)
    
    for _, row in sub.iterrows():
        ax.text(row.PC1 + 0.001, row.PC2 + 0.001, suku, fontsize=8, alpha=0.7)

ax.axhline(0, ls="--", color="gray", alpha=0.3, lw=0.8)
ax.axvline(0, ls="--", color="gray", alpha=0.3, lw=0.8)
ax.set_xlabel(f"PC1 ({expvar[0]:.1f}%)")
ax.set_ylabel(f"PC2 ({expvar[1]:.1f}%)")
ax.set_title("PCA Perempuan Berdasarkan Suku Asal (Seluruh Angkatan)", fontweight="bold", fontsize=13)

ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="Suku Asal")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_FIG}/04c_pca_female_suku.png", dpi=150, bbox_inches="tight")
plt.show()


## 📐 Uji Statistik

### Metode yang digunakan:
| Uji | Fungsi |
|-----|--------|
| **Permutation MANOVA** (Procrustes ANOVA) | Uji signifikansi efek sex & tahun pada bentuk wajah |
| **Independent t-test** | Uji perbedaan PC1/centroid size antara sex |
| **Kruskal-Wallis** | Uji non-parametrik perbedaan antar tahun |


### UJI STATISTIK (Permutation MANOVA)

In [ ]:
# ── Permutation MANOVA (Procrustes ANOVA) ─────────────────────────────────────
def perm_manova(X, groups, n_perm=999, seed=42):
    """
    Permutation MANOVA: uji apakah grup berbeda secara multivariate.
    Statistik: F = SS_between/SS_within (mirip Anderson 2001 / adonis2 di R).
    """
    rng = np.random.default_rng(seed)
    groups = np.asarray(groups)
    uniq = np.unique(groups)
    n = len(groups)

    def fstat(X, g):
        grand = X.mean(0)
        ss_tot = ((X - grand)**2).sum()
        ss_w = sum(((X[g==u] - X[g==u].mean(0))**2).sum() for u in np.unique(g))
        ss_b = ss_tot - ss_w
        df_b, df_w = len(np.unique(g))-1, n-len(np.unique(g))
        return (ss_b/df_b) / (ss_w/df_w) if df_w>0 and ss_w>0 else 0

    F_obs = fstat(X, groups)
    F_perm = [fstat(X, rng.permutation(groups)) for _ in range(n_perm)]
    p = (np.array(F_perm) >= F_obs).sum() / n_perm
    return F_obs, p


# Gunakan PC scores (10 PC pertama agar tidak singulier)
n_pc = min(10, scores.shape[1])
Xpc = scores[:, :n_pc]

print("="*60)
print("📊 HASIL UJI STATISTIK (Permutation MANOVA, 999x)")
print("="*60)

# 1. Efek jenis kelamin
F_sex, p_sex = perm_manova(Xpc, df.sex.values)
print(f"\n1️⃣  Efek JENIS KELAMIN (male vs female)")
print(f"   F = {F_sex:.4f},  p = {p_sex:.4f}  {'✅ Signifikan (p<0.05)' if p_sex<0.05 else '❌ Tidak signifikan'}")

# 2. Efek tahun
F_yr, p_yr = perm_manova(Xpc, df.year.values)
print(f"\n2️⃣  Efek TAHUN ({', '.join(YEARS)})")
print(f"   F = {F_yr:.4f},  p = {p_yr:.4f}  {'✅ Signifikan (p<0.05)' if p_yr<0.05 else '❌ Tidak signifikan'}")

# 3. T-test PC1 sex
m_pc1 = df[df.sex=="male"].PC1.values
f_pc1 = df[df.sex=="female"].PC1.values
if len(m_pc1)>1 and len(f_pc1)>1:
    t, pt = stats.ttest_ind(m_pc1, f_pc1)
    pool_std = np.sqrt((m_pc1.std()**2 + f_pc1.std()**2)/2)
    d = abs(m_pc1.mean()-f_pc1.mean()) / pool_std if pool_std>0 else 0
    print(f"\n3️⃣  T-test PC1 (male vs female)")
    print(f"   t = {t:.4f},  p = {pt:.4f},  Cohen's d = {d:.4f}")
    print(f"   Ukuran efek: {'Besar (>0.8)' if d>0.8 else 'Sedang (0.5-0.8)' if d>0.5 else 'Kecil (<0.5)'}")

# 4. Kruskal-Wallis centroid size antar tahun
yr_cs = [df[df.year==y].cs.values for y in YEARS]
yr_cs_valid = [v for v in yr_cs if len(v)>0]
if len(yr_cs_valid)>1:
    H, pkw = stats.kruskal(*yr_cs_valid)
    print(f"\n4️⃣  Kruskal-Wallis: Centroid Size antar Tahun")
    print(f"   H = {H:.4f},  p = {pkw:.4f}  {'✅ Signifikan' if pkw<0.05 else '❌ Tidak signifikan'}")

# 5. T-test centroid size sex
m_cs = df[df.sex=="male"].cs.values
f_cs = df[df.sex=="female"].cs.values
if len(m_cs)>1 and len(f_cs)>1:
    t2, pt2 = stats.ttest_ind(m_cs, f_cs)
    print(f"\n5️⃣  T-test: Centroid Size (male vs female)")
    print(f"   t = {t2:.4f},  p = {pt2:.4f}  {'✅ Signifikan' if pt2<0.05 else '❌ Tidak signifikan'}")
    print(f"   Mean male = {m_cs.mean():.4f}  ±{m_cs.std():.4f}")
    print(f"   Mean female = {f_cs.mean():.4f}  ±{f_cs.std():.4f}")

print("\n✅ Semua uji statistik selesai!")

#### Visualisasi Hasil UJI Statistik

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle("Ringkasan Analisis Statistik", fontsize=14, fontweight="bold")

# (0,0) Boxplot PC1 per sex
ax = axes[0,0]
data_bp = [df[df.sex==s].PC1.values for s in ["male","female"]]
bp = ax.boxplot(data_bp, patch_artist=True, notch=True,
                labels=["Laki-laki","Perempuan"],
                medianprops=dict(color="black",lw=2))
for patch, c in zip(bp["boxes"], [COLOR["male"],COLOR["female"]]):
    patch.set_facecolor(c); patch.set_alpha(0.7)
ax.set_title(f"PC1 per Jenis Kelamin\n(t-test p={pt:.4f})", fontweight="bold")
ax.set_ylabel("PC1 Score"); ax.grid(alpha=0.3, axis="y")

# (0,1) Violin centroid size per sex
ax = axes[0,1]
for i, (sex, c) in enumerate([("male",COLOR["male"]),("female",COLOR["female"])]):
    v = ax.violinplot(df[df.sex==sex].cs.values, positions=[i],
                      showmedians=True, showextrema=True)
    for part in v["bodies"]: part.set_facecolor(c); part.set_alpha(0.6)
    v["cmedians"].set_color("black"); v["cmedians"].set_lw(2)
ax.set_xticks([0,1]); ax.set_xticklabels(["Laki-laki","Perempuan"])
ax.set_title(f"Centroid Size per Jenis Kelamin\n(t-test p={pt2:.4f})", fontweight="bold")
ax.set_ylabel("Centroid Size"); ax.grid(alpha=0.3, axis="y")

# (1,0) Boxplot PC1 per tahun
ax = axes[1,0]
data_yr = [df[df.year==y].PC1.values for y in YEARS]
bp2 = ax.boxplot(data_yr, patch_artist=True, notch=False, labels=YEARS,
                 medianprops=dict(color="black",lw=2))
yr_pal = ["#FF6D00","#6200EA","#00BFA5"]
for patch, c in zip(bp2["boxes"], yr_pal):
    patch.set_facecolor(c); patch.set_alpha(0.7)
ax.set_title(f"PC1 per Tahun\n(Perm-MANOVA p={p_yr:.4f})", fontweight="bold")
ax.set_ylabel("PC1 Score"); ax.grid(alpha=0.3, axis="y")

# (1,1) Centroid size mean±SEM per tahun & sex
ax = axes[1,1]
x_pos = np.arange(len(YEARS))
for i, (sex, c, shift) in enumerate([("male",COLOR["male"],-0.18),("female",COLOR["female"],0.18)]):
    means = [df[(df.sex==sex)&(df.year==y)].cs.mean() for y in YEARS]
    sems  = [df[(df.sex==sex)&(df.year==y)].cs.sem()  for y in YEARS]
    means = [m if not np.isnan(m) else 0 for m in means]
    sems  = [s if not np.isnan(s) else 0 for s in sems]
    lbl = "Laki-laki" if sex=="male" else "Perempuan"
    ax.errorbar(x_pos+shift, means, yerr=sems, fmt="o", color=c,
                markersize=9, capsize=5, lw=2, label=lbl)
ax.set_xticks(x_pos); ax.set_xticklabels(YEARS)
ax.set_title("Centroid Size per Tahun & Jenis Kelamin\n(Mean ± SEM)", fontweight="bold")
ax.set_ylabel("Centroid Size"); ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUT_FIG}/05_stat_plots.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: 05_stat_plots.png")


# ## 🎯 Canonical Variate Analysis (CVA)
# CVA memaksimalkan pemisahan antar-grup dibanding PCA yang hanya melihat variasi total.

# In[28]:


from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# CVA: Sex (male vs female)
valid_sex = df[df.sex.isin(["male","female"])].copy()
if len(valid_sex.sex.unique()) >= 2:
    n_pc_cva = min(10, scores.shape[1])
    X_cva = scores[valid_sex.index, :n_pc_cva]

    lda = LinearDiscriminantAnalysis()
    cv_scores = lda.fit_transform(X_cva, valid_sex.sex.values)
    valid_sex["CV1"] = cv_scores[:, 0] if cv_scores.ndim > 1 else cv_scores

    fig, ax = plt.subplots(figsize=(10, 6))
    for sex, clr in [("male", COLOR["male"]), ("female", COLOR["female"])]:
        sub = valid_sex[valid_sex.sex == sex]
        label = "Laki-laki" if sex == "male" else "Perempuan"
        ax.hist(sub.CV1, bins=12, alpha=0.6, color=clr, label=label, edgecolor="white")

    acc = lda.score(X_cva, valid_sex.sex.values) * 100
    ax.set_title(f"CVA — Laki-laki vs Perempuan (Akurasi: {acc:.1f}%)",
                 fontweight="bold", fontsize=13)
    ax.set_xlabel("Canonical Variate 1"); ax.set_ylabel("Frekuensi")
    ax.legend(fontsize=11); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{OUT_FIG}/06_cva.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✅ CVA Akurasi klasifikasi: {acc:.1f}%")
    print(f"✅ Saved: {OUT_FIG}/06_cva.png")
else:
    print("⚠️ CVA butuh minimal 2 grup sex yang terdeteksi")


# ## 🌐 Principal Coordinate Analysis (PCoA)
# PCoA berbasis Procrustes distance matrix — pelengkap PCA.

# In[29]:


from scipy.spatial.distance import pdist, squareform

# Hitung Procrustes distance matrix
n_spec = len(df)
proc_dist = np.zeros((n_spec, n_spec))
for i in range(n_spec):
    for j in range(i+1, n_spec):
        diff = df.aligned.iloc[i] - df.aligned.iloc[j]
        d = np.sqrt((diff**2).sum())
        proc_dist[i,j] = proc_dist[j,i] = d

# PCoA via eigendecomposition
H = np.eye(n_spec) - np.ones((n_spec, n_spec)) / n_spec
B = -0.5 * H @ (proc_dist**2) @ H
eigvals, eigvecs = np.linalg.eigh(B)
idx = np.argsort(eigvals)[::-1]
eigvals, eigvecs = eigvals[idx], eigvecs[:, idx]

# Ambil 2 axis pertama (positif)
pos = eigvals > 0
pcoa1 = eigvecs[:, 0] * np.sqrt(eigvals[0])
pcoa2 = eigvecs[:, 1] * np.sqrt(eigvals[1])
var1 = eigvals[0] / eigvals[pos].sum() * 100
var2 = eigvals[1] / eigvals[pos].sum() * 100

fig, ax = plt.subplots(figsize=(10, 8))
for sex, clr in [("male", COLOR["male"]), ("female", COLOR["female"])]:
    mask = df.sex == sex
    label = "Laki-laki" if sex == "male" else "Perempuan"
    ax.scatter(pcoa1[mask], pcoa2[mask], c=clr, s=80, alpha=0.8,
               edgecolors="white", lw=0.5, label=label, zorder=3)

ax.set_xlabel(f"PCoA 1 ({var1:.1f}%)"); ax.set_ylabel(f"PCoA 2 ({var2:.1f}%)")
ax.set_title("PCoA — Procrustes Distance", fontweight="bold", fontsize=13)
ax.axhline(0, ls="--", color="gray", alpha=0.3)
ax.axvline(0, ls="--", color="gray", alpha=0.3)
ax.legend(fontsize=11); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_FIG}/07_pcoa.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Saved: {OUT_FIG}/07_pcoa.png")

In [ ]:

# Export deskriptif statistik
summary = df.groupby(["year","sex"]).agg(
    N       =("PC1","count"),
    PC1_mean=("PC1","mean"),
    PC1_std =("PC1","std"),
    PC2_mean=("PC2","mean"),
    PC2_std =("PC2","std"),
    CS_mean =("cs","mean"),
    CS_std  =("cs","std")
).round(4)
summary.to_csv(f"{OUT_TBL}/deskriptif_statistik.csv")

# Export PCA variance
pca_var = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(min(15, len(expvar)))],
    "Variance_%": expvar[:min(15, len(expvar))].round(4),
    "Cumulative_%": np.cumsum(expvar[:min(15, len(expvar))]).round(4)
})
pca_var.to_csv(f"{OUT_TBL}/pca_variance.csv", index=False)

# Export PC scores
pc_export = df[["year","sex","image_name","PC1","PC2","PC3","cs"]].copy()
pc_export.to_csv(f"{OUT_TBL}/pc_scores.csv", index=False)

print(f"✅ Exported ke {OUT_TBL}/:")
for f in os.listdir(OUT_TBL):
    print(f"   📄 {f}")


# ## 📋 Ringkasan Hasil

# In[ ]:


print("="*65)
print("  RINGKASAN ANALISIS GEOMETRIK MORFOMETRIK WAJAH — FaceDig TPS")
print("="*65)

print("\n📌 STATISTIK DESKRIPTIF PER GRUP")
summary = df.groupby(["year","sex"]).agg(
    N       =("PC1","count"),
    PC1_mean=("PC1","mean"),
    PC2_mean=("PC2","mean"),
    CS_mean =("cs","mean"),
    CS_std  =("cs","std")
).round(4)
print(summary.to_string())

print("\n\n📌 VARIANS PCA (10 PC pertama)")
print(f"  {'PC':<6} {'% Var':>8} {'Kumulatif':>12}")
cumul = 0
for i in range(min(10,len(expvar))):
    cumul += expvar[i]
    bar = "█" * int(expvar[i]/2)
    print(f"  PC{i+1:<4} {expvar[i]:>7.2f}%  {cumul:>10.2f}%  {bar}")

print("\n\n📌 HASIL UJI STATISTIK")
print(f"  Efek jenis kelamin : F={F_sex:.4f}, p={p_sex:.4f}"
      f"  {'✅ Signifikan' if p_sex<0.05 else '❌ Tidak signifikan'}")
print(f"  Efek tahun         : F={F_yr:.4f}, p={p_yr:.4f}"
      f"  {'✅ Signifikan' if p_yr<0.05 else '❌ Tidak signifikan'}")

print("\n📁 File output yang dihasilkan:")
for f_out in [f"{OUT_FIG}/01_landmark_foto.png",f"{OUT_FIG}/02_mean_shape.png",
              f"{OUT_FIG}/03_pca_sex.png",f"{OUT_FIG}/04_pca_year_sex.png",
              f"{OUT_FIG}/05_stat_plots.png",
              f"{OUT_FIG}/06_cva.png",f"{OUT_FIG}/07_pcoa.png",
              f"{OUT_FIG}/08_heatmap.png"]:
    exists = "✅" if os.path.exists(f_out) else "⬜"
    print(f"  {exists} {f_out}")
print("\n✅ Selesai!")

